**Delta Table Hands-on Exercises**

In [0]:
%sql
CREATE OR REPLACE TABLE courier_parcels_delta (
parcel_id INT, customer_name STRING, destination_city STRING, service_type STRING,
weight_kg DOUBLE, shipping_fee INT, delivery_status STRING
) USING DELTA;

In [0]:
%sql
INSERT INTO courier_parcels_delta VALUES
(101,'Arjun','Hyderabad','Express',2.5,500,'Delivered'),
(102,'Meera','Mumbai','Standard',4.0,350,'In Transit'),
(103,'Karan','Delhi','Express',1.5,450,'Pending'),
(104,'Sneha','Pune','Standard',3.0,300,'Delivered'),
(105,'Rohit','Chennai','Express',5.0,800,'In Transit'),
(106,'Divya','Hyderabad','Standard',2.0,250,'Cancelled'),
(107,'Nikhil','Bengaluru','Express',6.0,950,'Delivered'),
(108,'Pooja','Mumbai','Standard',1.0,150,'Pending'),
(109,'Vikram','Delhi','Express',3.5,650,'In Transit'),
(110,'Anjali','Pune','Standard',4.5,400,'Cancelled'),
(111,'Farhan','Chennai','Express',2.0,550,'Delivered'),
(112,'Kavya','Hyderabad','Standard',5.5,600,'Pending'),
(113,'Manoj','Bengaluru','Express',1.5,400,'In Transit'),
(114,'Riya','Delhi','Standard',3.0,300,'Delivered'),
(115,'Tarun','Mumbai','Express',4.0,700,'Pending');

num_affected_rows,num_inserted_rows
15,15


In [0]:
%sql
SELECT * FROM courier_parcels_delta ORDER BY parcel_id; 

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
101,Arjun,Hyderabad,Express,2.5,500,Delivered
102,Meera,Mumbai,Standard,4.0,350,In Transit
103,Karan,Delhi,Express,1.5,450,Pending
104,Sneha,Pune,Standard,3.0,300,Delivered
105,Rohit,Chennai,Express,5.0,800,In Transit
106,Divya,Hyderabad,Standard,2.0,250,Cancelled
107,Nikhil,Bengaluru,Express,6.0,950,Delivered
108,Pooja,Mumbai,Standard,1.0,150,Pending
109,Vikram,Delhi,Express,3.5,650,In Transit
110,Anjali,Pune,Standard,4.5,400,Cancelled


In [0]:
%sql
CREATE OR REPLACE TABLE courier_parcels_copy USING DELTA AS SELECT * FROM courier_parcels_delta; 

num_affected_rows,num_inserted_rows


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
schema = StructType([
    StructField("parcel_id", IntegerType(), False),
    StructField("customer_name", StringType(), True),
    StructField("destination_city", StringType(), True),
    StructField("service_type", StringType(), True),
    StructField("weight_kg", DoubleType(), True),
    StructField("shipping_fee", IntegerType(), True),
    StructField("delivery_status", StringType(), True)
])
rows = [
    (201, "Ishaan", "Jaipur", "Express", 2.2, 520, "Pending"),
    (202, "Priya", "Kolkata", "Standard", 3.8, 340, "In Transit"),
    (203, "Zoya", "Bengaluru", "Express", 1.7, 480, "Delivered")
]
parcel_df = spark.createDataFrame(rows, schema)
display(parcel_df)
parcel_df.write.format("delta").mode("overwrite").saveAsTable("courier_parcels_pyspark")

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
201,Ishaan,Jaipur,Express,2.2,520,Pending
202,Priya,Kolkata,Standard,3.8,340,In Transit
203,Zoya,Bengaluru,Express,1.7,480,Delivered


In [0]:
%sql
DESCRIBE DETAIL courier_parcels_delta; 
DESCRIBE DETAIL courier_parcels_copy;
DESCRIBE DETAIL courier_parcels_pyspark;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,f331e58f-a5c2-4c46-9e49-7ce8f1bd38bd,kimoo_red.default.courier_parcels_pyspark,null,,2026-11-05T09:22:20.487Z,2026-11-05T09:22:21.000Z,List(),List(),1,2389,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
INSERT INTO courier_parcels_delta VALUES (116,'Sameer','Jaipur','Express',2.5,550,'Pending'); 

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
UPDATE courier_parcels_delta SET delivery_status='Delivered' WHERE parcel_id=103; 

num_affected_rows
1


In [0]:
%sql
UPDATE courier_parcels_delta SET shipping_fee=shipping_fee+100 WHERE service_type='Express'; 

num_affected_rows
9


In [0]:
%sql
UPDATE courier_parcels_delta SET destination_city='Hyderabad' WHERE parcel_id=108; 

num_affected_rows
1


In [0]:
%sql
DELETE FROM courier_parcels_delta WHERE delivery_status='Cancelled'; 

num_affected_rows
2


In [0]:
%sql
SELECT * FROM courier_parcels_delta ORDER BY shipping_fee DESC, parcel_id; 

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
107,Nikhil,Bengaluru,Express,6.0,1050,Delivered
105,Rohit,Chennai,Express,5.0,900,In Transit
115,Tarun,Mumbai,Express,4.0,800,Pending
109,Vikram,Delhi,Express,3.5,750,In Transit
111,Farhan,Chennai,Express,2.0,650,Delivered
116,Sameer,Jaipur,Express,2.5,650,Pending
101,Arjun,Hyderabad,Express,2.5,600,Delivered
112,Kavya,Hyderabad,Standard,5.5,600,Pending
103,Karan,Delhi,Express,1.5,550,Delivered
113,Manoj,Bengaluru,Express,1.5,500,In Transit


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_parcels AS
SELECT * FROM VALUES
(102,'Meera','Mumbai','Express',4.0,650,'Delivered'),
(105,'Rohit','Chennai','Express',5.0,900,'Delivered'),
(117,'Ayesha','Kolkata','Standard',2.0,300,'Pending'),
(118,'Rahul','Jaipur','Express',3.0,600,'In Transit')
AS incoming(parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status);

In [0]:
%sql
SELECT * FROM incoming_parcels ORDER BY parcel_id; 

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
102,Meera,Mumbai,Express,4.0,650,Delivered
105,Rohit,Chennai,Express,5.0,900,Delivered
117,Ayesha,Kolkata,Standard,2.0,300,Pending
118,Rahul,Jaipur,Express,3.0,600,In Transit


In [0]:
%sql
SELECT i.parcel_id FROM incoming_parcels i INNER JOIN courier_parcels_delta t
ON i.parcel_id=t.parcel_id ORDER BY i.parcel_id; 

parcel_id
102
105


In [0]:
%sql
SELECT i.parcel_id FROM incoming_parcels i LEFT ANTI JOIN courier_parcels_delta t
ON i.parcel_id=t.parcel_id ORDER BY i.parcel_id; 

parcel_id
117
118


In [0]:
%sql
DESCRIBE HISTORY courier_parcels_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
8,2026-11-05T09:24:40.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3958517962211808),86accde9-b883-456e-b456-4256c163563d,1105-084803-2cvkfpxo-v2n,7,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 6516, p25FileSize -> 3524, numDeletionVectorsRemoved -> 1, minFileSize -> 3524, numAddedFiles -> 1, maxFileSize -> 3524, p75FileSize -> 3524, p50FileSize -> 3524, numAddedBytes -> 3524)",null,Databricks-Runtime/19.13.x-photon-scala2.13
7,2026-11-05T09:24:39.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(delivery_status#16003 = Cancelled)""])",null,List(3958517962211808),86accde9-b883-456e-b456-4256c163563d,1105-084803-2cvkfpxo-v2n,6,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1027, numDeletionVectorsUpdated -> 1, numDeletedRows -> 2, scanTimeMs -> 616, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 409)",null,Databricks-Runtime/19.13.x-photon-scala2.13
6,2026-11-05T09:24:25.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(parcel_id#15398 = 108)""])",null,List(3958517962211808),e54d5e4c-0f15-4d98-80f1-0af87ae096ad,1105-084803-2cvkfpxo-v2n,5,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1900, numDeletionVectorsUpdated -> 0, scanTimeMs -> 607, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 2976, rewriteTimeMs -> 1293)",null,Databricks-Runtime/19.13.x-photon-scala2.13
5,2026-11-05T09:24:12.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3958517962211808),db8b737c-f7ad-470b-b74d-eda15e2e0657,1105-084803-2cvkfpxo-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 5961, p25FileSize -> 3540, numDeletionVectorsRemoved -> 1, minFileSize -> 3540, numAddedFiles -> 1, maxFileSize -> 3540, p75FileSize -> 3540, p50FileSize -> 3540, numAddedBytes -> 3540)",null,Databricks-Runtime/19.13.x-photon-scala2.13
4,2026-11-05T09:24:10.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(service_type#14567 = Express)""])",null,List(3958517962211808),db8b737c-f7ad-470b-b74d-eda15e2e0657,1105-084803-2cvkfpxo-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 5143, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1927, numDeletionVectorsUpdated -> 1, scanTimeMs -> 604, numAddedFiles -> 1, numUpdatedRows -> 9, numAddedBytes -> 3336, rewriteTimeMs -> 1322)",null,Databricks-Runtime/19.13.x-photon-scala2.13
3,2026-11-05T09:23:54.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(parcel_id#13963 = 103)""])",null,List(3958517962211808),32dd8f05-3516-4e0c-abf7-7d6e51bb7424,1105-084803-2cvkfpxo-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2221, numDeletionVectorsUpdated -> 0, scanTimeMs -> 616, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 2959, rewriteTimeMs -> 1600)",null,Databricks-Runtime/19.13.x-photon-scala2.13
2,2026-11-05T09:23:29.000Z,14497686701437

In [0]:
%sql
MERGE INTO courier_parcels_delta AS t
USING incoming_parcels AS s
ON t.parcel_id=s.parcel_id
WHEN MATCHED THEN UPDATE SET
t.customer_name=s.customer_name,
t.destination_city=s.destination_city,
t.service_type=s.service_type,
t.weight_kg=s.weight_kg,
t.shipping_fee=s.shipping_fee,
t.delivery_status=s.delivery_status
WHEN NOT MATCHED THEN INSERT
(parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status)
VALUES (s.parcel_id,s.customer_name,s.destination_city,s.service_type,s.weight_kg,s.shipping_fee,s.delivery_status); 

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,2,0,2


In [0]:
%sql
SELECT * FROM courier_parcels_delta WHERE parcel_id IN (102,105) ORDER BY parcel_id; 

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
102,Meera,Mumbai,Express,4.0,650,Delivered
105,Rohit,Chennai,Express,5.0,900,Delivered


In [0]:
%sql
SELECT * FROM courier_parcels_delta WHERE parcel_id IN (117,118) ORDER BY parcel_id; 

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
117,Ayesha,Kolkata,Standard,2.0,300,Pending
118,Rahul,Jaipur,Express,3.0,600,In Transit


In [0]:
%sql
DESCRIBE DETAIL courier_parcels_delta; 

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,448b3644-50d0-4d13-b31d-453355f9879a,kimoo_red.default.courier_parcels_delta,null,,2026-11-05T09:21:35.844Z,2026-11-05T09:26:27.000Z,List(),List(),1,3579,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-797be4dc-e1e2-4a27-be96-91b15918a7f5, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-8cd46aa3-42a7-40ab-aa50-2e6706a2bf19)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE HISTORY courier_parcels_delta; 

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
10,2026-11-05T09:26:27.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3958517962211808),86ac381e-8f2c-4f4c-b27a-aef44ffcf5bc,1105-084803-2cvkfpxo-v2n,9,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 15190, p25FileSize -> 3579, numDeletionVectorsRemoved -> 1, minFileSize -> 3579, numAddedFiles -> 1, maxFileSize -> 3579, p75FileSize -> 3579, p50FileSize -> 3579, numAddedBytes -> 3579)",null,Databricks-Runtime/19.13.x-photon-scala2.13
9,2026-11-05T09:26:26.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(parcel_id#17154 = parcel_id#17133)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3958517962211808),86ac381e-8f2c-4f4c-b27a-aef44ffcf5bc,1105-084803-2cvkfpxo-v2n,8,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 11666, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 2739, materializeSourceTimeMs -> 118, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 894, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1701)",null,Databricks-Runtime/19.13.x-photon-scala2.13
8,2026-11-05T09:24:40.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3958517962211808),86accde9-b883-456e-b456-4256c163563d,1105-084803-2cvkfpxo-v2n,7,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 6516, p25FileSize -> 3524, numDeletionVectorsRemoved -> 1, minFileSize -> 3524, numAddedFiles -> 1, maxFileSize -> 3524, p75FileSize -> 3524, p50FileSize -> 3524, numAddedBytes -> 3524)",null,Databricks-Runtime/19.13.x-photon-scala2.13
7,2026-11-05T09:24:39.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(delivery_status#16003 = Cancelled)""])",null,List(3958517962211808),86accde9-b883-456e-b456-4256c163563d,1105-084803-2cvkfpxo-v2n,6,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1027, numDeletionVectorsUpdated -> 1, numDeletedRows -> 2, scanTimeMs -> 616, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 409)",null,Databricks-Runtime/19.13.x-photon-scala2.13
6,2026-11-05T09:24:25.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(parcel_id#15398 = 108)""])",null,List(3958517962211808),e54d5e4c-0f15-4d98-80f1-0af87ae096ad,1105-084803-2cvkfpxo-v2n,5,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1900, numDeletionVectorsUpdated -> 0, scanTimeMs -> 607, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 2976, rewriteTimeMs -> 1293)",null,Databricks-Runtime/19.13.x-photon-scala2.13
5,2026-11-05T09:24:12.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy

In [0]:
display(_sqldf)

In [0]:
%sql
SELECT *
FROM courier_parcels_delta VERSION AS OF 6
ORDER BY parcel_id;

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
101,Arjun,Hyderabad,Express,2.5,600,Delivered
102,Meera,Mumbai,Standard,4.0,350,In Transit
103,Karan,Delhi,Express,1.5,550,Delivered
104,Sneha,Pune,Standard,3.0,300,Delivered
105,Rohit,Chennai,Express,5.0,900,In Transit
106,Divya,Hyderabad,Standard,2.0,250,Cancelled
107,Nikhil,Bengaluru,Express,6.0,1050,Delivered
108,Pooja,Hyderabad,Standard,1.0,150,Pending
109,Vikram,Delhi,Express,3.5,750,In Transit
110,Anjali,Pune,Standard,4.5,400,Cancelled


In [0]:
%sql
SELECT *
FROM courier_parcels_delta VERSION AS OF 6
WHERE parcel_id = 102;

SELECT *
FROM courier_parcels_delta
WHERE parcel_id = 102;

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
102,Meera,Mumbai,Express,4.0,650,Delivered


In [0]:
%sql
OPTIMIZE courier_parcels_delta; 

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1793870872751, 1793870873287, 8, 0, null, List(0, 0), null, 7, 7, 0, 0, null, null, 0)"


In [0]:
%sql
DESCRIBE DETAIL courier_parcels_delta; 

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,448b3644-50d0-4d13-b31d-453355f9879a,kimoo_red.default.courier_parcels_delta,null,,2026-11-05T09:21:35.844Z,2026-11-05T09:26:27.000Z,List(),List(),1,3579,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-797be4dc-e1e2-4a27-be96-91b15918a7f5, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-8cd46aa3-42a7-40ab-aa50-2e6706a2bf19)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
OPTIMIZE courier_parcels_delta ZORDER BY (destination_city);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 3579), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1793870902817, 1793870903302, 8, 0, null, List(0, 0), null, 7, 7, 0, 0, null, null, 0)"


In [0]:
%sql
DESCRIBE HISTORY courier_parcels_delta; 

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
10,2026-11-05T09:26:27.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3958517962211808),86ac381e-8f2c-4f4c-b27a-aef44ffcf5bc,1105-084803-2cvkfpxo-v2n,9,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 15190, p25FileSize -> 3579, numDeletionVectorsRemoved -> 1, minFileSize -> 3579, numAddedFiles -> 1, maxFileSize -> 3579, p75FileSize -> 3579, p50FileSize -> 3579, numAddedBytes -> 3579)",null,Databricks-Runtime/19.13.x-photon-scala2.13
9,2026-11-05T09:26:26.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(parcel_id#17154 = parcel_id#17133)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3958517962211808),86ac381e-8f2c-4f4c-b27a-aef44ffcf5bc,1105-084803-2cvkfpxo-v2n,8,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 11666, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 2739, materializeSourceTimeMs -> 118, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 894, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1701)",null,Databricks-Runtime/19.13.x-photon-scala2.13
8,2026-11-05T09:24:40.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3958517962211808),86accde9-b883-456e-b456-4256c163563d,1105-084803-2cvkfpxo-v2n,7,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 6516, p25FileSize -> 3524, numDeletionVectorsRemoved -> 1, minFileSize -> 3524, numAddedFiles -> 1, maxFileSize -> 3524, p75FileSize -> 3524, p50FileSize -> 3524, numAddedBytes -> 3524)",null,Databricks-Runtime/19.13.x-photon-scala2.13
7,2026-11-05T09:24:39.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(delivery_status#16003 = Cancelled)""])",null,List(3958517962211808),86accde9-b883-456e-b456-4256c163563d,1105-084803-2cvkfpxo-v2n,6,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1027, numDeletionVectorsUpdated -> 1, numDeletedRows -> 2, scanTimeMs -> 616, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 409)",null,Databricks-Runtime/19.13.x-photon-scala2.13
6,2026-11-05T09:24:25.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(parcel_id#15398 = 108)""])",null,List(3958517962211808),e54d5e4c-0f15-4d98-80f1-0af87ae096ad,1105-084803-2cvkfpxo-v2n,5,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1900, numDeletionVectorsUpdated -> 0, scanTimeMs -> 607, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 2976, rewriteTimeMs -> 1293)",null,Databricks-Runtime/19.13.x-photon-scala2.13
5,2026-11-05T09:24:12.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy